**Upload the CSV File**

In [29]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore (1).csv


**Read the CSV**

In [30]:
import pandas as pd

df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

In [31]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Create a SQLite Database Connection**





In [37]:
import sqlite3

conn = sqlite3.connect("superstore.db")

In [33]:
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)

9994

**Verify the Table Was Created**

In [34]:
query = "SELECT * FROM superstore_raw LIMIT 5"

pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Create customers Table**

In [44]:
conn.execute("""
CREATE TABLE IF NOT EXISTS customers AS
SELECT DISTINCT
    "Customer ID" AS customer_id,
    "Customer Name" AS customer_name,
    Segment,
    Country,
    City,
    State,
    Region
FROM superstore_raw;
""")

In [39]:
pd.read_sql("SELECT * FROM customers LIMIT 5", conn)

,customer_id,customer_name,Segment,Country,City,State,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South


**Create products Table**


In [45]:
conn.execute("""
CREATE TABLE IF NOT EXISTS products AS
SELECT DISTINCT
    "Product ID" AS product_id,
    "Product Name" AS product_name,
    Category,
    "Sub-Category" AS sub_category
FROM superstore_raw;
""")

In [ ]:
pd.read_sql("SELECT * FROM products LIMIT 5", conn)

**Create orders Table**

In [46]:
conn.execute("""
CREATE TABLE  IF NOT EXISTS orders AS
SELECT DISTINCT
    "Order ID" AS order_id,
    "Order Date" AS order_date,
    "Ship Date" AS ship_date,
    "Ship Mode" AS ship_mode,
    "Customer ID" AS customer_id,
    "Product ID" AS product_id,
    Sales,
    Quantity,
    Discount,
    Profit
FROM superstore_raw;
""")

In [47]:
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164


**Check Record Counts**

In [48]:
print(pd.read_sql("SELECT COUNT(*) AS customers FROM customers", conn))

print(pd.read_sql("SELECT COUNT(*) AS products FROM products", conn))

print(pd.read_sql("SELECT COUNT(*) AS orders FROM orders", conn))

   customers
0       4688
   products
0      1894
   orders
0    9993


**Customers with Above-Average Sales**

In [49]:
query = """
SELECT
    customer_id,
    SUM(Sales) AS total_sales
FROM orders
GROUP BY customer_id
HAVING SUM(Sales) > (
    SELECT AVG(customer_total)
    FROM (
        SELECT  SUM(Sales) AS customer_total
        FROM orders
        GROUP BY customer_id
    )
)
ORDER BY total_sales DESC;
"""

pd.read_sql(query, conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
289,JK-16120,2932.484
290,SW-20455,2921.544
291,ML-17410,2921.500
292,RD-19585,2912.894


This query calculates the total  sales of each customer.

It then computes the average sales across all customers.

Finally, it returns only those customers whose total sales exceed that average.



**Highest Order for Each Customer**

In [50]:
query = """
SELECT
    customer_id,
    order_id,
    Sales
FROM orders o
WHERE Sales = (
    SELECT MAX(Sales)
    FROM orders
    WHERE customer_id = o.customer_id
)
ORDER BY customer_id;
"""

pd.read_sql(query, conn)

,customer_id,order_id,Sales
0,AA-10315,CA-2016-103982,3930.072
1,AA-10375,CA-2016-131065,499.980
2,AA-10480,CA-2016-114601,479.970
3,AA-10645,CA-2015-110863,1323.900
4,AB-10015,CA-2016-140935,341.960
...,...,...,...
790,XP-21865,CA-2014-114335,337.088
791,YC-21895,CA-2014-119375,2934.330
792,YS-21880,CA-2017-119809,2793.528
793,ZC-21910,CA-2015-115567,1516.200


The subquery finds the maximum order value for each customer.

The outer query returns the orders matching that maximum value.

This helps identify each customer's largest purchase.

**Total Sales per Customer using CTE**

In [51]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT *
FROM customer_sales
ORDER BY total_sales DESC;
"""

pd.read_sql(query, conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
788,RS-19870,22.328
789,MG-18205,16.739
790,CJ-11875,16.520
791,LD-16855,5.304


The CTE customer_sales calculates total sales for each customer.

The main query simply reads from the CTE.

**Window Function (ROW_NUMBER)**

In [52]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_sales,
    ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS row_num
FROM customer_sales;
"""

pd.read_sql(query, conn)

,customer_id,total_sales,row_num
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


ROW_NUMBER() gives a unique sequential rank.

Even if two customers have the same sales, they get different row numbers.

**Rank Customers by Total Sales**

In [53]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_sales,
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM customer_sales;
"""

pd.read_sql(query, conn)

,customer_id,total_sales,sales_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


RANK() assigns the same rank to tied values.

If two customers share Rank 1, the next rank will be 3.

**JOIN + CTE + Window Function**

In [55]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_id,
    c.customer_name,
    cs.total_sales,
    RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
FROM customer_sales cs
JOIN customers c
    ON cs.customer_id = c.customer_id
ORDER BY sales_rank;
"""

pd.read_sql(query, conn)

,customer_id,customer_name,total_sales,sales_rank
0,SM-20320,Sean Miller,25043.050,1
1,SM-20320,Sean Miller,25043.050,1
2,SM-20320,Sean Miller,25043.050,1
3,SM-20320,Sean Miller,25043.050,1
4,SM-20320,Sean Miller,25043.050,1
...,...,...,...,...
4683,MG-18205,Mitch Gastineau,16.739,4684
4684,CJ-11875,Carl Jackson,16.520,4685
4685,LD-16855,Lela Donovan,5.304,4686
4686,TS-21085,Thais Sissman,4.833,4687


CTE calculates total sales per customer.

JOIN fetches customer names.

RANK() ranks customers by revenue contribution.

**Top 10 Customers**

In [54]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT *
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
5,KL-16645,14175.229
6,SC-20095,14142.334
7,HL-15040,12873.298
8,SE-20110,12209.438
9,CC-12370,12129.072


**Bottom 10 Customers**

In [56]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT *
FROM customer_sales
ORDER BY total_sales ASC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_id,total_sales
0,TS-21085,4.833
1,LD-16855,5.304
2,CJ-11875,16.520
3,MG-18205,16.739
4,RS-19870,22.328
5,SG-20890,47.946
6,RE-19405,48.360
7,LB-16735,50.188
8,AS-10135,58.820
9,JC-15340,71.263


**Customers with Only One Order**

In [57]:
query = """
SELECT
    customer_id,
    COUNT(order_id) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(order_id) = 1;
"""

pd.read_sql(query, conn)

,customer_id,order_count
0,AO-10810,1
1,CJ-11875,1
2,JR-15700,1
3,LD-16855,1
4,RE-19405,1
